# Installing Libraries

In [ ]:
%pip install torch torchvision pillow gradio

# Final Script

# Gradio App

In [30]:
import torch
from transformers import ViTForImageClassification, ViTImageProcessor
from PIL import Image

# =========================
# 1. Load model
# =========================

model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=38,
    ignore_mismatched_sizes=True
)

state_dict = torch.load("plant_vit.pth", map_location="cpu")

model.load_state_dict(state_dict)
model.eval()

# =========================
# 2. Load image processor
# =========================

processor = ViTImageProcessor.from_pretrained(
    "google/vit-base-patch16-224"
)

# =========================
# 3. Load image
# =========================

image = Image.open("pepper_bell_bacterial_spot.jpeg").convert("RGB")

# =========================
# 4. Preprocess image
# =========================

inputs = processor(images=image, return_tensors="pt")

# =========================
# 5. Predict
# =========================

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
predicted_class_idx = logits.argmax(-1).item()
print("")
print("")
print("Predicted class index:", predicted_class_idx)

from joblib import load
dataset=load("dataset.joblib")
print("Predicted class name:",dataset.classes[predicted_class_idx])

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([38, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([38])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5b2d822fd734b83b01.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Rough work

In [ ]:
import torch
from transformers import ViTForImageClassification, ViTImageProcessor
from PIL import Image

# =========================
# 1. Load model
# =========================

model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=38,
    ignore_mismatched_sizes=True
)

state_dict = torch.load("plant_vit.pth", map_location="cpu")

model.load_state_dict(state_dict)
model.eval()

# =========================
# 2. Load image processor
# =========================

processor = ViTImageProcessor.from_pretrained(
    "google/vit-base-patch16-224"
)

# =========================
# 3. Load image
# =========================

image = Image.open("pepper_bell_bacterial_spot.jpeg").convert("RGB")

# =========================
# 4. Preprocess image
# =========================

inputs = processor(images=image, return_tensors="pt")

# =========================
# 5. Predict
# =========================

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
predicted_class_idx = logits.argmax(-1).item()
print("")
print("")
print("Predicted class index:", predicted_class_idx)

from joblib import load
dataset=load("dataset.joblib")
print("Predicted class name:",dataset.classes[predicted_class_idx])

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([38, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([38])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.




Predicted class index: 18
Predicted class name: Pepper,_bell___Bacterial_spot


In [ ]:
import gradio as gr
import torch
from transformers import ViTForImageClassification, ViTImageProcessor
from PIL import Image

# ==========================================
# Load model
# ==========================================

MODEL_PATH = "plant_vit.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=38,
    ignore_mismatched_sizes=True
)

state_dict = torch.load(MODEL_PATH, map_location=device)

model.load_state_dict(state_dict)

model.to(device)
model.eval()

# ==========================================
# Image processor
# ==========================================

processor = ViTImageProcessor.from_pretrained(
    "google/vit-base-patch16-224"
)

# ==========================================
# Class labels
# ==========================================

class_names = [
    "Apple___Apple_scab",
    "Apple___Black_rot",
    "Apple___Cedar_apple_rust",
    "Apple___healthy",
    "Blueberry___healthy",
    "Cherry___Powdery_mildew",
    "Cherry___healthy",
    "Corn___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn___Common_rust",
    "Corn___Northern_Leaf_Blight",
    "Corn___healthy",
    "Grape___Black_rot",
    "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
    "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)",
    "Peach___Bacterial_spot",
    "Peach___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Raspberry___healthy",
    "Soybean___healthy",
    "Squash___Powdery_mildew",
    "Strawberry___Leaf_scorch",
    "Strawberry___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy"
]

# ==========================================
# Prediction function
# ==========================================

def predict(image):

    inputs = processor(images=image, return_tensors="pt")

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_class_idx = logits.argmax(-1).item()

    prediction = class_names[predicted_class_idx]

    confidence = torch.softmax(logits, dim=-1)[0][predicted_class_idx].item()

    return {
        prediction: confidence
    }

# ==========================================
# Gradio UI
# ==========================================

title = "🌿 Plant Disease Classification"

description = """
Upload a plant leaf image to classify plant diseases using a Vision Transformer (ViT) model.
"""

interface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=3),
    title=title,
    description=description,
    examples=[
        ["pepper_bell_bacterial_spot.jpeg"]
    ]
)

# ==========================================
# Launch app
# ==========================================

if __name__ == "__main__":
    interface.launch()

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([38, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([38])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fd81c4f7812fa48bcd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

def greet(name, intensity):
    return "Hello, " + name + "!" * int(intensity)

demo = gr.Interface(
    fn=greet,
    inputs=["text", "slider"],
    outputs=["text"],
    api_name="predict"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3c632f4e611e8a0f41.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import torch
from transformers import ViTForImageClassification

# Create the model architecture
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    ignore_mismatched_sizes=True,
    num_labels=38
)

# Load your .pth weights
state_dict = torch.load("plant_vit.pth", map_location="cpu")

model.load_state_dict(state_dict)

model.eval()

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([38, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([38])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermed

In [ ]:
import torch
from transformers import ViTForImageClassification, ViTImageProcessor
from PIL import Image

# =========================
# 2. Load image processor
# =========================

processor = ViTImageProcessor.from_pretrained(
    "google/vit-base-patch16-224"
)

# =========================
# 3. Load image
# =========================

image = Image.open("pepper_bell_bacterial_spot.jpeg").convert("RGB")

# =========================
# 4. Preprocess image
# =========================

inputs = processor(images=image, return_tensors="pt")

# =========================
# 5. Predict
# =========================

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
predicted_class_idx = logits.argmax(-1).item()

print("Predicted class index:", predicted_class_idx)
from joblib import load
dataset=load("dataset.joblib")
print(dataset.classes[predicted_class_idx])

Predicted class index: 18
Pepper,_bell___Bacterial_spot


# Get requirements.txt

In [29]:
%pip freeze>requirements.txt